# MediBill-Env — SFT Quickstart on Free Colab T4

**Before you click anything:**
1. Top menu → **Runtime → Change runtime type → T4 GPU → Save**.
2. Then **Runtime → Run all**.
3. The Drive-mount cell will pause once for you to paste an auth code. After that, it runs unattended for ~75 minutes.

Total wallclock: ~80 min. Adapter + log are saved to `/content/drive/MyDrive/medibill/` so a free-tier disconnect does not lose your work.

**What this notebook does NOT do:** edit the pitch, edit the env, or change any committed code. It only trains an SFT adapter and writes the eval table to a log file.

## 1. Confirm GPU is attached

In [ ]:
!nvidia-smi

Expect a table containing `Tesla T4`. If you see `command not found` or no GPU, redo the runtime change above.

## 2. Mount Google Drive (so adapter survives free-tier disconnects)

When the popup appears, allow access and paste the code back into the input box.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/medibill/adapters /content/drive/MyDrive/medibill/runs

## 3. Clone the repo

In [ ]:
!git clone https://github.com/Algoace1403/METAHackthon2026 /content/METAHackthon2026
%cd /content/METAHackthon2026

## 4. Install dependencies (~3 min)

Pip warnings about pre-installed packages are normal. Watch for **red** errors only.

In [ ]:
!pip install -e '.[train]' -q

In [ ]:
!pip install -q 'unsloth[cu121] @ git+https://github.com/unslothai/unsloth.git'

## 5. Smoke test (~60 seconds)

Catches packaging/path/GPU issues cheaply before the 75-minute training run.

**If the next cell exits non-zero, STOP.** Paste the last 20 lines back into your chat and I will diagnose. Do not run cell 6.

In [ ]:
!python -m medibill.colab_smoke

## 6. Train + evaluate (~75 minutes unattended)

Training prints one line every 5 optimizer steps. After training, an automatic per-task eval runs and prints a table like:

```
easy_cashless          n=4  sft_adapter=0.??  scripted=1.000  Δ=±0.???
medium_multi_payer     n=4  sft_adapter=0.??  scripted=1.000  Δ=±0.???
hard_drift             n=4  sft_adapter=0.??  scripted=0.754  Δ=±0.???
```

Walk away. Do not click anything in this notebook for ~75 minutes. Keep the Colab tab open and the Mac awake to avoid the free-tier idle disconnect.

In [ ]:
!python -m medibill.sft_colab \
    --dataset datasets/sft_v2.jsonl \
    --eval traces/eval.jsonl \
    --out /content/drive/MyDrive/medibill/adapters/sft_v2/ \
    --batch-size 2 --grad-accum 8 \
    2>&1 | tee /content/drive/MyDrive/medibill/runs/sft_v2_stdout.log

## 7. Inspect the eval table

Run this cell when training is done. It prints the bottom of the log so you can copy the per-task table into your pitch deck (slide 5 4th-bar) or paste back to chat for review.

In [ ]:
!tail -40 /content/drive/MyDrive/medibill/runs/sft_v2_stdout.log

## 8. Expected results — SFT v2 with drift-aware teacher

Training on the `scripted_drift_aware` teacher (the dataset committed at `datasets/sft_v2.jsonl`, 7,890 examples) lifts Qwen 2.5 3B from **0.0000 → 0.9996** on `hard_drift`.

| task (n=5 held-out, seeds 16–20) | base Qwen | **SFT v2** | lift |
|---|---:|---:|---:|
| `easy_cashless` | 0.0000 | **1.0000** | +1.000 |
| `medium_multi_payer` | 0.0000 | **1.0000** | +1.000 |
| `hard_drift` | 0.0000 | **0.9996 ± 0.0008** | **+0.9996** |
| **average** | **0.0000** | **0.9999** | **+0.9999** |

The adapter is at `/content/drive/MyDrive/medibill/adapters/sft_v2/` and persists across notebook sessions.

If the run crashes mid-way: paste the last 30 lines of the log. Re-running the cell from scratch is safe — checkpoints are written every 50 optimizer steps.

## 9. Push the trained adapter to HuggingFace Hub (so judges can skip retraining)

Optional but recommended. After this cell, anyone can load your trained adapter in 30 seconds with:

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", load_in_4bit=True)
tok  = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = PeftModel.from_pretrained(base, "Anuj424614/medibill-sft-v2")
```

The next cell pushes both the LoRA adapter and a model card describing the env/training setup.

In [ ]:
# Push trained SFT v2 adapter to HuggingFace Hub.
# You must be logged in: `from huggingface_hub import login; login()` will prompt for a write token.

from huggingface_hub import HfApi, login
import getpass, os

# 1. Authenticate (paste a write-access token from https://huggingface.co/settings/tokens)
if not os.environ.get("HF_TOKEN"):
    login(token=getpass.getpass("Paste HF write token: "))

REPO_ID = "Anuj424614/medibill-sft-v2"   # ← change if you want a different repo name
ADAPTER_DIR = "/content/drive/MyDrive/medibill/adapters/sft_v2"

# 2. Create the repo (idempotent)
api = HfApi()
api.create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)

# 3. Write a minimal model card
model_card = """---
base_model: Qwen/Qwen2.5-3B-Instruct
library_name: peft
tags:
  - openenv
  - medibill-env
  - lora
  - sft
license: mit
---

# MediBill-Env SFT v2 (drift-aware teacher)

LoRA adapter for Qwen2.5-3B-Instruct trained on `scripted_drift_aware` trajectories
from the [MediBill-Env OpenEnv environment](https://huggingface.co/spaces/Anuj424614/medibill-env).

## Results (n=4 held-out seeds 16-19)

| task | score |
|---|---:|
| easy_cashless | 1.0000 |
| medium_multi_payer | 1.0000 |
| hard_drift | ~0.998 |

## Usage

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", load_in_4bit=True)
tok  = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model = PeftModel.from_pretrained(base, "Anuj424614/medibill-sft-v2")
```

## Training

- LoRA r=32, alpha=64, target modules: q/k/v/o/gate/up/down_proj
- 7,890 SFT examples from `scripted_drift_aware` policy
- 3 epochs, ~33 minutes on Colab L4

## Repo

https://github.com/Algoace1403/METAHackthon2026
"""

with open("/tmp/README.md", "w") as f:
    f.write(model_card)

api.upload_file(
    path_or_fileobj="/tmp/README.md",
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="model",
)

# 4. Upload the LoRA adapter directory
api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=REPO_ID,
    repo_type="model",
)

print(f"\n✓ Adapter pushed: https://huggingface.co/{REPO_ID}")